# Part 5: Build an agent over the knowledge base

Part 5 at Caldova begins when the supply-chain team wants an agent to reason over evidence without asking the knowledge base to synthesize an answer. You will create an extractive knowledge base from the Part 1 knowledge sources, connect Microsoft Agent Framework to its MCP endpoint, then let the agent work with the retrieved evidence.

**🎯 Mission**
- Create a model-free knowledge base configured for extractive output
- Connect Microsoft Agent Framework to the knowledge base through its MCP endpoint
- Run an operational query and inspect the agent's tool activity

## How to read this notebook

Follow **section map > code > Compare with the product**. The orange box marks the current focus. Blue boxes provide context. Section arrows show a learning sequence, not a live deployment status.

Start with the [service roles and full learning path](README.md). Genuine portal screenshots appear only at matching checkpoints. These architecture maps are diagrams, not screenshots. Use the portal paths and actual service/query evidence to inspect your own environment.


## Where you are in the architecture

This exercise returns to the three Part 1 sources. It does not connect the combined Part 4 KB to an agent. The new extractive KB has no attached model. Your local Agent Framework process calls its MCP endpoint and uses Azure OpenAI to synthesize the answer. No hosted Foundry agent is deployed.

[![Part 5 architecture: a local agent calls an extractive knowledge base through MCP, then synthesizes the answer.](../../img/architecture/azure-agent.png)](../../img/architecture/azure-agent.png)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

**Read the map:** orange marks focused objects, blue arrows show retrieval, purple arrows show model calls, and gray cards are not connected to the focused KB. These are intended connections, not verified deployment states. The checkpoints below read your real service configuration without changing it.

The lab has separate KB variants. Part 1 branches into Part 2 with Fabric and Part 3 with Work IQ. Part 4 combines them. Part 5 reuses the Part 1 sources.

See the [visual field guide](../../docs/visual-guide.md) for portal navigation, expected observations, and the screenshot reference gallery.

## Step 1: Set up the agent environment

Load the Azure OpenAI model and Foundry IQ settings from the project `.env` file. These settings create the extractive knowledge base and connect it to the Agent Framework model.

### Section map

![Part 5, step 1: Local agent configuration is loaded from the private environment.](../../img/architecture/step-configuration.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
import os

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework_openai import OpenAIChatClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
MODEL_DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-minimal-knowledge-base"
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"

print("Agent environment loaded")

### Open your resources

Run this read-only helper cell to get links built from your local lab configuration. Keep Azure Portal, Fabric, and the notebook side by side. A sign-in page or an access-denied screen is not evidence that a resource is missing.

If the helper import fails, select the lab's `.venv` kernel and use `src/notebooks` as the notebook working directory. Do not copy credentials into the helper or screenshots.

In [ ]:
from lab_visuals import (
    show_activity,
    show_architecture,
    show_file_ingestion,
    show_portal_links,
    show_references,
    verify_knowledge_base,
    verify_sources,
)

show_portal_links(
    AZURE_SEARCH_SERVICE_ENDPOINT,
    search_resource_id=os.getenv("AZURE_SEARCH_RESOURCE_ID", ""),
)

### Compare with the product

**Expected after this section:** Local agent configuration is loaded from the private environment.

**Where to look:** Microsoft Foundry > existing model deployment and Azure Portal > Search service.

**What to verify:** The agent code runs locally. No hosted Foundry agent is deployed.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 2: Add authentication to MCP requests

The knowledge base MCP endpoint accepts the Azure AI Search admin key that is preconfigured for this lab. Define a header provider that adds the key to each MCP request.

### Section map

![Part 5, step 2: A local header provider is defined for MCP authentication.](../../img/architecture/step-agent-auth.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
def get_mcp_headers(_):
    return {"api-key": AZURE_SEARCH_ADMIN_KEY}

### Compare with the product

**Expected after this section:** A local header provider is defined for MCP authentication.

**Where to look:** The notebook function definition, without invoking or printing secret headers.

**What to verify:** Defining a function makes no remote request. Never screenshot the admin key or the header values.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Concept: The important change is who writes

This KB uses minimal retrieval reasoning and extractive output, with no attached chat model. It returns evidence over MCP. The local Agent Framework agent calls its own model and composes the answer. This is not a hosted Foundry agent, and the KB contains only the three Part 1 sources.

[![Parts 1 to 4 use KB answer synthesis, while Part 5 returns extracts to a local agent that writes the answer.](../../img/concepts/answer-boundary.png)](../../img/concepts/answer-boundary.png)

*Concept illustration, not a portal screenshot or a successful query result.*


## Step 3: Create an extractive knowledge base

Create a new knowledge base over the three knowledge sources from Part 1. Minimal reasoning effort does not perform source selection, query planning, or iterative search. `EXTRACTIVE_DATA` returns retrieved evidence without attaching a model or synthesizing a knowledge base answer.

### Section map

![Part 5, step 3: A new extractive KB connects only the three Part 1 sources.](../../img/architecture/step-agent-kb.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    "file-docs-knowledge-source",
    "supplier-evidence-knowledge-source",
    "sourcing-documents-knowledge-source",
]

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova extractive knowledge base for supply-chain evidence",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Extractive knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated")

### See what you built: retrieval without KB synthesis

**Portal check:** open **Agentic retrieval > Knowledge bases** and select `caldova-supply-chain-minimal-knowledge-base`.

**Expected:** the three Part 1 sources, `extractiveData` output, minimal reasoning, and no attached model. Fabric and Work IQ are intentionally absent.

**Think about it:** the Azure OpenAI model still exists, but which component will call it now?

In [ ]:
verify_knowledge_base(
    index_client, KNOWLEDGE_BASE_NAME, knowledge_source_names, extractive=True
)

### Compare with the product

**Expected after this section:** A new extractive KB connects only the three Part 1 sources.

**Where to look:** Search > Agentic retrieval > Knowledge bases > caldova-supply-chain-minimal-knowledge-base.

**What to verify:** extractiveData output, minimal reasoning, no attached model. Use read-back if the portal does not expose a setting.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge bases > caldova-supply-chain-minimal-knowledge-base.

**Look for:** In **Basics**, count only the three Part 1 sources. Expand **Retrieval** and **Output configurations** to inspect **Minimal** reasoning, **Extractive data**, and the empty **Chat completion model** section.

[![Original Azure portal view: Extractive knowledge base source membership.](../../img/portal/extractive-knowledge-base-sources.png)](../../img/portal/extractive-knowledge-base-sources.png)

[![Original Azure portal view: Minimal reasoning and extractive output settings.](../../img/portal/extractive-knowledge-base-settings.png)](../../img/portal/extractive-knowledge-base-settings.png)

**How to read this:** The Add model deployment button is expected here. Do not add a KB model. Search returns evidence, while the local agent will use its own model to write the answer. These are two views of the same deployed KB.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 4: Create the knowledge base MCP tool

Every Foundry IQ knowledge base exposes a streamable HTTP MCP endpoint. Build the extractive knowledge base endpoint and expose only the `knowledge_base_retrieve` operation to the agent.

### Section map

![Part 5, step 4: A local MCP tool points at the KB's existing endpoint.](../../img/architecture/step-agent-kb.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
knowledge_base_endpoint = (
    f"{AZURE_SEARCH_SERVICE_ENDPOINT.rstrip('/')}"
    f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    f"?api-version={AZURE_SEARCH_API_VERSION}"
)
knowledge_base_mcp_tool = MCPStreamableHTTPTool(
    name="caldova-knowledge-base",
    url=knowledge_base_endpoint,
    header_provider=get_mcp_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
)

print(f"MCP tool configured for {knowledge_base_endpoint}")

### Compare with the product

**Expected after this section:** A local MCP tool points at the KB's existing endpoint.

**Where to look:** The notebook tool configuration and the matching KB name in Search.

**What to verify:** Only knowledge_base_retrieve is allowed. No separately deployed MCP server or hosted agent is created by this step.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 5: Create the supply-chain agent

Create a code-first Agent Framework agent backed by the Azure OpenAI model deployed for the lab. The knowledge base returns extracts without synthesis, while the agent reasons over that evidence to produce the final answer.

### Section map

![Part 5, step 5: An Agent Framework Python object uses the existing Azure OpenAI deployment.](../../img/architecture/step-agent-run.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
openai_client = AsyncOpenAI(
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=AZURE_OPENAI_KEY,
)
chat_client = OpenAIChatClient(
    model=MODEL_DEPLOYMENT_NAME,
    async_client=openai_client,
)
agent = Agent(
    client=chat_client,
    name="CaldovaSupplyChainAgent",
    instructions=(
        "You are a Caldova supply-chain assistant. "
        "Always use the Caldova knowledge base tool for supply-chain questions. "
        "Ground every answer in retrieved evidence, cite the source titles or identifiers, "
        "and keep policy, approved commitments, and operational evidence distinct. "
        "If the retrieved evidence is insufficient, say what information is unavailable."
    ),
    tools=[knowledge_base_mcp_tool],
    default_options={"store": False},
)

print(f"Agent '{agent.name}' created")

### See what you built: a local agent, not a cloud deployment

`CaldovaSupplyChainAgent` is a Python object in this notebook's process. There is no new hosted agent to find in the Foundry portal.

**Inspect instead:** compare the MCP endpoint with the extractive KB name. In Microsoft Foundry, inspect the existing `gpt-5.4` deployment. After the next query, use the tool activity below to see the request the agent actually sent.

**Expected:** the agent reasons with Azure OpenAI, calls the Search KB over MCP, and receives evidence rather than a prewritten KB answer.

### Compare with the product

**Expected after this section:** An Agent Framework Python object uses the existing Azure OpenAI deployment.

**Where to look:** The notebook agent object and Microsoft Foundry > model deployments.

**What to verify:** The cloud model exists, but CaldovaSupplyChainAgent is local. Do not look for a new hosted agent in Foundry.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 6: Run a query

Open the MCP connection for one query, then close it automatically when the response completes. Change `question` and rerun this cell to ask additional supply-chain questions without leaving an MCP lifecycle task running.

### Section map

![Part 5, step 6: The local agent retrieves evidence through MCP and synthesizes a response.](../../img/architecture/step-agent-run.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)

async with knowledge_base_mcp_tool:
    response = await agent.run(question)

display(Markdown(response.text))

### Compare with the product

**Expected after this section:** The local agent retrieves evidence through MCP and synthesizes a response.

**Where to look:** The notebook answer, followed by the tool-call trace.

**What to verify:** The KB returns extracts. The agent calls Azure OpenAI for synthesis. The MCP connection is scoped to this query.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 7: Inspect agent activity

Agent responses include the messages produced during `run()`. Display each MCP tool call and its arguments to see the query the agent sent to the knowledge base before producing its final answer.

### Section map

![Part 5, step 7: The notebook exposes the actual tool calls used during the agent run.](../../img/architecture/step-evidence.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the agent phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-agent.svg)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Configure the local agent environment and authentication for the KB's MCP endpoint. No hosted Foundry agent is deployed.
2. Create a separate three-source KB using policy and the two document indexes. Set extractiveData, minimal reasoning and no KB-attached model.
3. Create the MCP tool client and the local Python agent with its Azure OpenAI model client.
4. Run a question and follow the solid blue agent-to-KB arrow. The agent invokes knowledge_base_retrieve over MCP and the KB retrieves evidence from its three sources.
5. Follow the purple arrow directly from the local agent to Azure OpenAI. The agent, not the KB, synthesizes the final answer from the returned extracts.
6. Inspect the answer and tool-call trace in the notebook. Fabric and Work IQ are gray and disconnected from this KB. The dashed gray policy embedding path belongs to earlier ingestion.

</details>


In [ ]:
import json

table_rows = []
for message in response.messages:
    for content in message.contents:
        if content.type == "function_call":
            arguments = content.arguments
            if not isinstance(arguments, str):
                arguments = json.dumps(arguments, indent=2)
            arguments = arguments.replace("|", "\\|").replace("\n", "<br>")
            table_rows.append((content.name, arguments))

table = ["| Call | Tool | Arguments |", "| ---: | --- | --- |"]
for call_number, (tool_name, arguments) in enumerate(table_rows, start=1):
    table.append(f"| {call_number} | `{tool_name}` | `{arguments}` |")

display(Markdown("\n".join(table)))

In [ ]:
if not any("knowledge_base_retrieve" in name for name, _ in table_rows):
    raise RuntimeError("No knowledge_base_retrieve call was recorded. This run has not verified KB grounding.")
display(Markdown(
    "**Observed:** the agent issued the tool calls above. Inspect the answer's source identifiers "
    "against the retrieved evidence. Tool activity alone does not establish that every answer claim is supported."
))

#### 🔍 Source Hunt

1. Find the `knowledge_base_retrieve` call and inspect its query.
2. Which source identifiers in the answer support the shipment excursion and distribution decision?
3. Which component synthesized the answer, and why does the KB have no attached model?
4. Why are Fabric and Work IQ absent from this agent's KB?

### Compare with the product

**Expected after this section:** The notebook exposes the actual tool calls used during the agent run.

**Where to look:** The notebook tool-call table and the cited source identifiers.

**What to verify:** Find knowledge_base_retrieve, inspect the query, and match the answer to evidence. This trace, not a portal agent list, shows the local agent's work.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Architecture checkpoint

[![Part 5 architecture recap: compare the local agent and MCP flow with the verified source inventory and query evidence.](../../img/architecture/azure-agent.png)](../../img/architecture/azure-agent.png)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

**Before moving on:** identify what was preconfigured, what this notebook created, and which sources actually returned evidence. The diagram is a learning guide, not a live health report.

## ✅ Mission complete

**What you built:**

* ✓ **Extractive knowledge base**: A model-free knowledge base returns evidence from the Part 1 sources without answer synthesis.
* ✓ **Authenticated MCP connection**: The agent reaches the extractive knowledge base with the lab's configured Search key.
* ✓ **Agent Framework supply-chain agent**: An Azure OpenAI-backed agent reasons over the retrieved evidence to answer an operational question.

➡️ Continue to the [lab summary](../../instructions/summary.md).